# Spatial field theory in **d = 2** — `compute_cumulants` vs simulation

This notebook runs a **2-D** stochastic field theory through the public
`compute_cumulants` API (tree level) and validates the resulting correlator
against (i) the exact closed form and (ii) a 2-D Langevin simulation.

Theory:  $\partial_t\phi = D\nabla^2\phi - \mu\phi + \eta$, $\langle\eta\eta\rangle = 2T\,\delta^2\delta$,
on the infinite plane.  The equal-time correlator has the exact form
$C(r,0) = \frac{T}{2\pi D}K_0(r\sqrt{\mu/D})$ (a 2-D screened/Yukawa correlator).

The backend computes the self-energy with the analytic Symanzik momentum
reduction (so the $\int d^2\ell$ loop is closed-form, **not** numerical angular
quadrature) and does the $q\to x$ inverse transform with the **radial/Hankel**
transform (here $J_0$ for d=2).

In [ ]:
import os, sys
import os, sys
# --- depth-robust repo root: walk up until we find the 'pipeline' package ---
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'pipeline')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(os.path.join(_root, 'notebooks'))  # cwd=notebooks/ so relative data paths resolve as before
import numpy as np
import matplotlib.pyplot as plt
from pipeline.compute import compute_cumulants
from pipeline.theory import TheoryBuilder
from msrjd.integration.spatial.spatial_correlator import free_correlator_static_closed_form as K0_exact
from models.spatial_field_2d_sim import (
    simulate_2d, radial_correlator_2d, radial_structure_factor_2d)

mu, D, T = 1.0, 1.0, 1.0

## 1. Build the d=2 theory and run `compute_cumulants` (tree level)

In [ ]:
model = (TheoryBuilder('linear field (d=2)', n_populations=0)
         .physical_field('phi', spatial_dim=2)        # <- d=2
         .parameter('mu', default=1.0, domain='positive')
         .parameter('D',  default=1.0, domain='positive')
         .parameter('T',  default=1.0, domain='positive')
         .equation(lhs='(Dt + mu - D*Laplacian)*phi', rhs='0')
         .set_action_text('phit*((Dt + mu - D*Laplacian)*phi) - T*phit^2')
         .boundary('infinite').initial('stationary').build())

rs = np.linspace(0.3, 4.0, 16)
tau_max, tau_step = 2.0, 0.5
out = compute_cumulants(model, k=2, max_ell=0,
        fundamental={'mu': mu, 'D': D, 'T': T},
        external_fields=[('phi', 1), ('phi', 1)], spatial_grid=rs,
        tau_max=tau_max, tau_step=tau_step, verbose=False,
        use_cache=False, mf_dae_n_starts=4)

tau_grid = np.arange(-tau_max, tau_max + tau_step * 0.5, tau_step)
C_tau_x = np.real(out['C_tau_x'])              # (n_tau, n_r)
i0 = int(np.argmin(np.abs(tau_grid)))          # the tau=0 row
C_r0 = C_tau_x[i0]
print('d=2 C(r, tau=0) via compute_cumulants:')
print(np.c_[rs, C_r0][:6])

## 2. Validate against the exact closed form $C(r,0)=\frac{T}{2\pi D}K_0(r\sqrt{\mu/D})$

In [ ]:
C_exact = np.array([K0_exact(r, mu, D, T, 2) for r in rs])
relerr = np.abs(C_r0 - C_exact) / C_exact
print('max rel error (r <= 3):', float(np.max(relerr[rs <= 3.0])))

plt.figure(figsize=(6, 4))
plt.plot(rs, C_r0, 'o', label='compute_cumulants (d=2)')
plt.plot(rs, C_exact, '-', label=r'exact $(T/2\pi D)\,K_0(r\sqrt{\mu/D})$')
plt.yscale('log'); plt.xlabel('r'); plt.ylabel('C(r, 0)')
plt.title('d=2 equal-time correlator: pipeline vs exact'); plt.legend()
plt.tight_layout(); plt.show()

## 3. Cross-check against a 2-D Langevin simulation

`models/spatial_field_2d_sim.py` integrates the same SPDE on a 64×64 periodic
grid (spectral exponential-Euler).  We compare the simulated real-space $C(r)$
and structure factor $S(q)$ to the theory.

In [ ]:
snaps, meta = simulate_2d(L=20.0, N=64, mu=mu, D=D, T=T,
                          n_steps=60000, burn_in=12000, record_every=15, seed=7)
rc, Cr_sim = radial_correlator_2d(snaps, meta, n_bins=40)
kc, Sq_sim = radial_structure_factor_2d(snaps, meta, n_bins=30)
print('2-D sim done; physical cutoff k_max = %.2f' % meta['k_max'])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(rc, Cr_sim, 'o', ms=4, label='2-D simulation')
ax[0].plot(rs, C_exact, '-', label=r'$K_0$ theory')
ax[0].set_xlim(0, 6); ax[0].set_yscale('log')
ax[0].set_xlabel('r'); ax[0].set_ylabel('C(r)')
ax[0].set_title('real-space correlator'); ax[0].legend()

ax[1].plot(kc, Sq_sim, 'o', ms=4, label='2-D simulation')
ax[1].plot(kc, T / (mu + D * kc ** 2), '-', label=r'$T/(\mu+Dq^2)$')
ax[1].set_xlim(0, 3); ax[1].set_xlabel('q'); ax[1].set_ylabel('S(q)')
ax[1].set_title('structure factor'); ax[1].legend()
plt.tight_layout(); plt.show()

band = (rc > 1.0) & (rc < 5.0)
Cr_th = np.array([K0_exact(r, mu, D, T, 2) for r in rc])
print('sim C(r) vs K0, median rel (r in [1,5]):',
      float(np.nanmedian(np.abs(Cr_sim[band] - Cr_th[band]) / Cr_th[band])))

## 4. The two-time correlator $C(r,\tau)$ (also from `compute_cumulants`)

In [ ]:
plt.figure(figsize=(6, 4))
for it in (i0, i0 + 1, i0 + 2, i0 + 3):
    plt.plot(rs, C_tau_x[it], 'o-', ms=3, label=f'tau = {tau_grid[it]:.1f}')
plt.yscale('log'); plt.xlabel('r'); plt.ylabel('C(r, tau)')
plt.title('d=2 two-time correlator'); plt.legend()
plt.tight_layout(); plt.show()

## Summary

A **d=2** field theory runs end-to-end through `compute_cumulants`:
the equal-time correlator matches the exact $K_0$ closed form (~1% at moderate
$r$) and the independent 2-D simulation (real-space $C(r)$ and $S(q)$ agree to
~3%).  The momentum loop is closed-form in any dimension (analytic Symanzik
reduction); the $q\to x$ step uses the radial/Hankel transform.

*Scope:* this is the **tree-level** d=2 path.  The 1-loop **bubble** in d>1 routes
through the C-stack (`sigma_parametric` at `spatial_dim=2` + `radial_inverse_ft`)
— validated at the building-block level (self-energy vs $\int d^2\ell$, transform
vs $K_0$); wiring it into `compute_cumulants(max_ell=1)` for d=2 is the next step.